In [3]:
import pandas as pd
import numpy as np
from typing import List, Optional, Dict, Any

def load_jsonl(path: str) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)

    # Stabilise les colonnes contenant dict/list
    for col in df.columns:
        try:
            if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                df[col] = df[col].apply(lambda x: str(x) if isinstance(x, (dict, list)) else x)
        except Exception:
            # Si une colonne est trop bizarre, on la stringify directement
            df[col] = df[col].astype("string")

    return df


def safe_numeric(s: pd.Series) -> pd.Series:
    """
    Convertit en numérique si possible.
    - Gère explicitement les bool en les castant en float.
    """
    # Bool dtype classique ou pandas BooleanDtype
    if pd.api.types.is_bool_dtype(s):
        return s.astype("float")

    # Certains objets peuvent contenir des bool au milieu d'autres types
    # -> conversion générale
    out = pd.to_numeric(s, errors="coerce")

    # Si la conversion produit un dtype bool (rare mais possible), cast float
    if pd.api.types.is_bool_dtype(out):
        out = out.astype("float")

    return out


def compute_feature_report(
    df: pd.DataFrame,
    feature: str,
    label_col: str = "label",
    top_k: int = 5
) -> Dict[str, Any]:
    s = df[feature]
    n = len(s)

    # Missing
    missing = int(s.isna().sum())
    missing_pct = (missing / n) * 100 if n else 0.0

    # Unique
    try:
        n_unique = int(s.nunique(dropna=True))
    except Exception:
        n_unique = int(pd.Series(s.astype(str)).nunique(dropna=True))

    # Version numérique
    s_num = safe_numeric(s)
    num_ratio = float(s_num.notna().mean()) if n else 0.0

    # Heuristique de type :
    # - bool traité comme numérique
    # - sinon numérique si >90% convertible
    if pd.api.types.is_bool_dtype(s):
        inferred_type = "boolean"
    else:
        inferred_type = "numeric" if num_ratio > 0.9 else "categorical/other"

    report: Dict[str, Any] = {
        "feature": feature,
        "inferred_type": inferred_type,
        "dtype": str(s.dtype),
        "n_rows": n,
        "n_unique": n_unique,
        "missing": missing,
        "missing_pct": float(missing_pct),
    }

    # Stats numériques (inclut bool => déjà cast float)
    if inferred_type in {"numeric", "boolean"}:
        non_na = s_num.dropna()

        if len(non_na) > 0:
            # on force en float pour éviter toute bizarrerie de dtype
            non_na = non_na.astype(float)

            # Stats robustes
            try:
                report.update({
                    "mean": float(non_na.mean()),
                    "std": float(non_na.std(ddof=1)) if len(non_na) > 1 else 0.0,
                    "min": float(non_na.min()),
                    "p25": float(non_na.quantile(0.25)),
                    "median": float(non_na.quantile(0.50)),
                    "p75": float(non_na.quantile(0.75)),
                    "max": float(non_na.max()),
                })
            except Exception:
                # En dernier recours, on ne casse pas le pipeline
                report.update({
                    "mean": None, "std": None, "min": None,
                    "p25": None, "median": None, "p75": None, "max": None
                })
        else:
            report.update({
                "mean": None, "std": None, "min": None,
                "p25": None, "median": None, "p75": None, "max": None
            })

    # Catégorielles
    else:
        s_cat = s.astype("string")
        vc = s_cat.value_counts(dropna=False).head(top_k)
        report["top_values"] = [(str(idx), int(val)) for idx, val in vc.items()]

    # Corrélation avec label (si label num)
    if label_col in df.columns and feature != label_col:
        y_num = safe_numeric(df[label_col])
        y_ratio = float(y_num.notna().mean()) if len(y_num) else 0.0

        # Corr uniquement si label est (quasi) numérique
        if y_ratio > 0.9 and inferred_type in {"numeric", "boolean"}:
            tmp = pd.DataFrame({"x": s_num, "y": y_num}).dropna()
            if len(tmp) >= 3 and tmp["x"].nunique() > 1 and tmp["y"].nunique() > 1:
                report["corr_with_label"] = float(tmp["x"].corr(tmp["y"]))
            else:
                report["corr_with_label"] = None
        else:
            report["corr_with_label"] = None

    return report


def print_report(report: Dict[str, Any]):
    feat = report["feature"]
    print("=" * 90)
    print(f"Feature: {feat}")
    print(f"Type inféré: {report.get('inferred_type')} | dtype brut: {report.get('dtype')}")
    print(f"Lignes: {report.get('n_rows')}")
    print(f"Valeurs différentes (hors NaN): {report.get('n_unique')}")
    print(f"NaN: {report.get('missing')} ({report.get('missing_pct'):.2f}%)")

    if report.get("inferred_type") in {"numeric", "boolean"}:
        print("Stats numériques:")
        for k in ["mean", "std", "min", "p25", "median", "p75", "max"]:
            print(f"  - {k}: {report.get(k)}")
    else:
        print("Top valeurs:")
        for val, cnt in report.get("top_values", []):
            print(f"  - {val}: {cnt}")

    if "corr_with_label" in report:
        print(f"Corrélation avec label: {report.get('corr_with_label')}")
    print("=" * 90)
    print()


def review_features(
    path: str = "../Data/train.jsonl",
    label_col: str = "label",
    batch_size: int = 10,
    top_k: int = 5,
    liste_features: Optional[List[str]] = None,
    exclude: Optional[List[str]] = None
):
    """
    Parcourt les features par batch.
    - Si liste_features est fournie, respecte cet ordre strictement.
    - Sinon, utilise l'ordre des colonnes du dataset.
    """
    df = load_jsonl(path)

    if exclude is None:
        exclude = []

    # Base features selon ordre demandé
    if liste_features is not None:
        # On garde seulement celles qui existent réellement dans df
        feature_cols = [f for f in liste_features if f in df.columns]
    else:
        feature_cols = list(df.columns)

    # Retire label + excludes
    feature_cols = [c for c in feature_cols if c != label_col and c not in exclude]

    print(f"Dataset chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
    if label_col not in df.columns:
        print(f"⚠️ Colonne label '{label_col}' introuvable. Corrélations ignorées.")
    print(f"Nombre de features à analyser: {len(feature_cols)}")
    print()

    for i in range(0, len(feature_cols), batch_size):
        batch = feature_cols[i:i + batch_size]
        print("#" * 90)
        print(f"Batch {i//batch_size + 1} | Features {i+1} à {i+len(batch)} / {len(feature_cols)}")
        print("#" * 90)
        print()

        for feature in batch:
            rep = compute_feature_report(df, feature, label_col=label_col, top_k=top_k)
            print_report(rep)

    # Renvoie df si tu veux l'exploiter après
    return df


In [6]:
import json
import pandas as pd

def load_jsonl_flat(path: str) -> pd.DataFrame:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    # Aplatit les dict imbriqués en colonnes avec séparateur "."
    df = pd.json_normalize(records, sep=".")
    return df

def drop_parent_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = set(df.columns)
    parents = set()

    for c in cols:
        # si une colonne est un préfixe exact d'autres colonnes enfants
        prefix = c + "."
        if any(other.startswith(prefix) for other in cols):
            parents.add(c)

    if parents:
        print(f"Colonnes 'parents' ignorées automatiquement ({len(parents)}):")
        # print trié pour lisibilité
        for p in sorted(parents)[:50]:
            print(" -", p)
        if len(parents) > 50:
            print(" ...")

    return df.drop(columns=list(parents), errors="ignore")

def review_features_with_flat_twitter(
    path: str = "../Data/train.jsonl",
    label_col: str = "label",
    batch_size: int = 10,
    top_k: int = 5,
    liste_features=None,
    exclude=None,
):
    # 1) Load + flatten
    df = load_jsonl_flat(path)

    # 2) Retire les colonnes parents redondantes
    df = drop_parent_columns(df)

    if exclude is None:
        exclude = []

    # 3) Ordre imposé si liste fournie
    if liste_features is not None:
        feature_cols = [f for f in liste_features if f in df.columns]
    else:
        feature_cols = [c for c in df.columns if c != label_col]

    # 4) Retire label + excludes
    feature_cols = [c for c in feature_cols if c != label_col and c not in exclude]

    print(f"Dataset: {df.shape[0]} lignes, {df.shape[1]} colonnes")
    print(f"Features à analyser: {len(feature_cols)}")
    if label_col not in df.columns:
        print(f"⚠️ label '{label_col}' absent -> corrélations ignorées.")
    print()

    # Ici tu réutilises compute_feature_report / print_report de la version corrigée
    for i in range(0, len(feature_cols), batch_size):
        batch = feature_cols[i:i+batch_size]
        print("#" * 90)
        print(f"Batch {i//batch_size + 1} | {i+1} à {i+len(batch)} / {len(feature_cols)}")
        print("#" * 90)
        print()

        for feature in batch:
            rep = compute_feature_report(df, feature, label_col=label_col, top_k=top_k)
            print_report(rep)

    return df


In [9]:
liste_features = ['quoted_status.extended_tweet.entities.urls',
'quoted_status.extended_tweet.entities.hashtags',
'quoted_status.extended_tweet.entities.user_mentions',
'quoted_status.extended_tweet.entities.symbols',
'quoted_status.extended_tweet.full_text',
'quoted_status.extended_tweet.display_text_range',
'quoted_status.in_reply_to_status_id_str',
'quoted_status.in_reply_to_status_id',
'quoted_status.created_at',
'quoted_status.in_reply_to_user_id_str',
'quoted_status.source',
'quoted_status.retweet_count',
'quoted_status.retweeted',
'quoted_status.geo',
'quoted_status.filter_level',
'quoted_status.in_reply_to_screen_name',
'quoted_status.is_quote_status',
'quoted_status.id_str',
'quoted_status.in_reply_to_user_id',
'quoted_status.favorite_count',
'quoted_status.id',
'quoted_status.text',
'quoted_status.place',
'quoted_status.lang',
'quoted_status.quote_count',
'quoted_status.favorited',
'quoted_status.coordinates',
'quoted_status.truncated',
'quoted_status.reply_count',
'quoted_status.entities.urls',
'quoted_status.entities.hashtags',
'quoted_status.entities.user_mentions',
'quoted_status.entities.symbols',
'quoted_status.contributors',
'quoted_status.user.utc_offset',
'quoted_status.user.friends_count',
'quoted_status.user.profile_image_url_https',
'quoted_status.user.listed_count',
'quoted_status.user.profile_background_image_url',
'quoted_status.user.default_profile_image',
'quoted_status.user.favourites_count',
'quoted_status.user.description',
'quoted_status.user.created_at',
'quoted_status.user.is_translator',
'quoted_status.user.profile_background_image_url_https',
'quoted_status.user.protected',
'quoted_status.user.screen_name',
'quoted_status.user.id_str',
'quoted_status.user.profile_link_color',
'quoted_status.user.translator_type',
'quoted_status.user.id',
'quoted_status.user.geo_enabled',
'quoted_status.user.profile_background_color',
'quoted_status.user.lang',
'quoted_status.user.profile_sidebar_border_color',
'quoted_status.user.profile_text_color',
'quoted_status.user.verified',
'quoted_status.user.profile_image_url',
'quoted_status.user.time_zone',
'quoted_status.user.url',
'quoted_status.user.contributors_enabled',
'quoted_status.user.profile_background_tile',
'quoted_status.user.profile_banner_url',
'quoted_status.user.statuses_count',
'quoted_status.user.follow_request_sent',
'quoted_status.user.followers_count',
'quoted_status.user.profile_use_background_image',
'quoted_status.user.default_profile',
'quoted_status.user.following',
'quoted_status.user.name',
'quoted_status.user.location',
'quoted_status.user.profile_sidebar_fill_color',
'quoted_status.user.notifications',
'quoted_status_permalink.expanded',
'quoted_status_permalink.display',
'quoted_status_permalink.url',
'entities.urls',
'entities.hashtags',
'entities.user_mentions',
'entities.symbols',
'user.utc_offset',
'user.profile_image_url_https',
'user.listed_count',
'user.profile_background_image_url',
'user.default_profile_image',
'user.favourites_count',
'user.description',
'user.created_at',
'user.is_translator',
'user.profile_background_image_url_https',
'user.protected',
'user.profile_link_color',
'user.translator_type',
'user.geo_enabled',
'user.profile_background_color',
'user.lang',
'user.profile_sidebar_border_color',
'user.profile_text_color',
'user.profile_image_url',
'user.time_zone',
'user.url',
'user.contributors_enabled',
'user.profile_background_tile',
'user.profile_banner_url',
'user.statuses_count',
'user.follow_request_sent',
'user.profile_use_background_image',
'user.default_profile',
'user.following',
'user.location',
'user.profile_sidebar_fill_color',
'user.notifications',
'quoted_status',
'quoted_status_permalink',
'extended_tweet.entities.urls',
'extended_tweet.entities.hashtags',
'extended_tweet.entities.user_mentions',
'extended_tweet.entities.symbols',
'extended_tweet.full_text',
'extended_tweet.display_text_range',
'quoted_status.possibly_sensitive',
'quoted_status.extended_entities.media',
'quoted_status.entities.media',
'quoted_status.display_text_range',
'extended_tweet.extended_entities.media',
'extended_tweet.entities.media',
'quoted_status.extended_tweet.extended_entities.media',
'quoted_status.extended_tweet.entities.media',
'place.country_code',
'place.country',
'place.full_name',
'place.bounding_box.coordinates',
'place.bounding_box.type',
'place.place_type',
'place.name',
'place.id',
'place.url',
'entities.media',
'extended_entities.media',
'quoted_status.quoted_status_id',
'quoted_status.quoted_status_id_str',
'quoted_status.place.country_code',
'quoted_status.place.country',
'quoted_status.place.full_name',
'quoted_status.place.bounding_box.coordinates',
'quoted_status.place.bounding_box.type',
'quoted_status.place.place_type',
'quoted_status.place.name',
'quoted_status.place.id',
'quoted_status.place.url',
'quoted_status.scopes.followers',
'quoted_status.geo.coordinates',
'quoted_status.geo.type',
'quoted_status.coordinates.coordinates',
'quoted_status.coordinates.type',
'geo.coordinates',
'geo.type',
'coordinates.coordinates',
'coordinates.type',
'quoted_status.withheld_in_countries',
'label']

liste_features= ['user.friends_count',]

df = review_features_with_flat_twitter(
    path="../Data/train.jsonl",
    label_col="label",
    batch_size=10,
    top_k=5,
    liste_features=liste_features
)

Colonnes 'parents' ignorées automatiquement (6):
 - coordinates
 - geo
 - place
 - quoted_status.coordinates
 - quoted_status.geo
 - quoted_status.place
Dataset: 154914 lignes, 183 colonnes
Features à analyser: 0

